# 10 - Reader: Text vs Image Results

This notebook gives the final results of the reading half of the project, on the 120 test questions. It uses the labels of notebook 09 and compares the five settings.

- `gold_text` and `gold_image` give the model the evidence pages, as text or as images
- `retrieved_text` and `retrieved_image` give it the top 5 pages of fused + rerank, as text or as images
- `closed_book` gives it only the question

In [1]:
import json

import numpy as np
import pandas as pd

from financebench_rag.data import REPO, gold_pages, load_questions, load_split
from financebench_rag.grading import numeric_match

SETTINGS = ["gold_text", "gold_image", "retrieved_text", "retrieved_image", "closed_book"]

questions = {q["financebench_id"]: q for q in load_questions()}
test_ids = load_split()["test"]
answers = {s: json.load(open(REPO / "results" / f"test_answers_{s}.json")) for s in SETTINGS}
retrieved = {fid: [tuple(page) for page in ranked[:5]] for fid, ranked in json.load(open(REPO / "results" / "test_retrieval_hybrid.json")).items()}

labels = pd.read_csv(REPO / "results" / "test_labels.csv").pivot(index="financebench_id", columns="setting", values="label").loc[test_ids, SETTINGS]
correct = labels == "correct"
print(labels.shape)

(120, 5)


## 1. Accuracy per setting

Accuracy is the share of the 120 questions answered correctly. Refusals and cut-off answers count as not correct, and their shares are shown separately. The 95% confidence interval comes from a bootstrap with 10,000 resamples of the test questions, as in notebook 06.

In [2]:
rng = np.random.default_rng(0)


def bootstrap_ci(values, n=10000):
    """95% confidence interval of the mean, from n resamples with replacement."""
    means = rng.choice(values, size=(n, len(values))).mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return f"[{low:.3f}, {high:.3f}]"


pd.DataFrame([
    {
        "setting": s,
        "accuracy": round(correct[s].mean(), 3),
        "95% CI": bootstrap_ci(correct[s].to_numpy()),
        "refusal": round((labels[s] == "refusal").mean(), 3),
        "cut off": round((labels[s] == "cut off").mean(), 3),
    }
    for s in SETTINGS
])

,setting,accuracy,95% CI,refusal,cut off
0,gold_text,0.783,"[0.708, 0.850]",0.092,0.058
1,gold_image,0.800,"[0.725, 0.867]",0.067,0.092
2,retrieved_text,0.558,"[0.467, 0.650]",0.208,0.117
3,retrieved_image,0.567,"[0.475, 0.658]",0.200,0.083
4,closed_book,0.183,"[0.117, 0.250]",0.083,0.183


- Text and image reach a similar accuracy in both page settings, and their intervals overlap. 
- Both retrieved settings are well below the gold settings, and closed-book is the lowest as expected. 
- With retrieved pages, the model refuses more than twice as often as with the evidence pages.

## 2. Text against image

The two formats are compared question by question, since both answer the same questions. The table counts the questions only one of the two answers correctly, and gives the difference in accuracy (image minus text) with a bootstrap interval. If the interval includes 0, the difference could be chance.

The last two rows repeat the comparison on the questions where neither answer was cut off, so that the token limit cannot tilt the result.

In [3]:
def compare(first, second, keep=None):
    """Question-by-question comparison of two settings, optionally on a subset of questions."""
    a, b = correct[first], correct[second]
    if keep is not None:
        a, b = a[keep], b[keep]
    diff = b.astype(int).to_numpy() - a.astype(int).to_numpy()
    return {
        "first": first,
        "second": second,
        "questions": len(diff),
        "both correct": int((a & b).sum()),
        "only first": int((diff == -1).sum()),
        "only second": int((diff == 1).sum()),
        "difference": round(diff.mean(), 3),
        "95% CI": bootstrap_ci(diff),
    }


def finished(first, second):
    return (labels[first] != "cut off") & (labels[second] != "cut off")


pd.DataFrame([
    compare("gold_text", "gold_image"),
    compare("retrieved_text", "retrieved_image"),
    compare("gold_text", "gold_image", finished("gold_text", "gold_image")),
    compare("retrieved_text", "retrieved_image", finished("retrieved_text", "retrieved_image")),
])

,first,second,questions,both correct,only first,only second,difference,95% CI
0,gold_text,gold_image,120,88,6,8,0.017,"[-0.042, 0.075]"
1,retrieved_text,retrieved_image,120,58,9,10,0.008,"[-0.067, 0.075]"
2,gold_text,gold_image,107,88,2,6,0.037,"[-0.009, 0.093]"
3,retrieved_text,retrieved_image,100,58,6,4,-0.020,"[-0.080, 0.040]"


All 4 intervals include 0, so the difference between text and image could be due to chance. 

## 3. Evidence pages against retrieved pages

The same comparison between the evidence pages and the retrieved pages shows how much accuracy is lost when the model has to rely on search.

In [4]:
pd.DataFrame([
    compare("gold_text", "retrieved_text"),
    compare("gold_image", "retrieved_image"),
])

,first,second,questions,both correct,only first,only second,difference,95% CI
0,gold_text,retrieved_text,120,60,34,7,-0.225,"[-0.325, -0.125]"
1,gold_image,retrieved_image,120,60,36,8,-0.233,"[-0.333, -0.133]"


Retrieved pages lower accuracy in both formats, and the whole confidence interval is below 0, so this loss holds up with only 120 questions. It is much larger than the gap between text and image.

## 4. Search failure or reading failure

For the retrieved settings, the questions are split by whether an evidence page was among the top 5 retrieved pages. When it was, a wrong answer is a reading failure. When it was not, the model never saw the evidence, so a miss is a search failure. The table counts the labels in each group.

In [5]:
found = pd.Series({fid: bool(gold_pages(questions[fid]) & set(retrieved[fid])) for fid in test_ids})

rows = []
for s in ["retrieved_text", "retrieved_image"]:
    for in_top_5 in [True, False]:
        group = labels.loc[found == in_top_5, s]
        rows.append({
            "setting": s,
            "evidence in top 5": in_top_5,
            "questions": len(group),
            **group.value_counts().reindex(["correct", "wrong", "refusal", "cut off"], fill_value=0).to_dict(),
            "accuracy": round((group == "correct").mean(), 3),
        })
pd.DataFrame(rows)

,setting,evidence in top 5,questions,correct,wrong,refusal,cut off,accuracy
0,retrieved_text,True,74,53,7,6,8,0.716
1,retrieved_text,False,46,14,7,19,6,0.304
2,retrieved_image,True,74,56,9,4,5,0.757
3,retrieved_image,False,46,12,9,20,5,0.261


When search finds an evidence page, accuracy is close to the gold settings in section 1. When it does not, refusal is the most common answer. Of the questions the retrieved settings miss, about 60% had no evidence page in the top 5, so search causes more errors than reading.

## 5. Results by question type

Accuracy per setting, split by the official question type. The `questions` column gives the size of each group.

In [6]:
question_type = pd.Series({fid: questions[fid]["question_type"] for fid in test_ids})
by_type = correct.groupby(question_type).mean().round(3)
by_type.insert(0, "questions", question_type.value_counts())
by_type

setting,questions,gold_text,gold_image,retrieved_text,retrieved_image,closed_book
domain-relevant,40,0.75,0.775,0.500,0.575,0.350
metrics-generated,40,0.90,0.900,0.700,0.625,0.025
novel-generated,40,0.70,0.725,0.475,0.500,0.175


With the evidence pages, metrics questions are the easiest type in both formats. In closed book settings they are almost never answered, while domain questions reach 0.350 since many of them can be answered from general knowledge, as *Islam et al. (2023)* also note.

## 6. Cost per question

The average input length, answer length and time per question, from notebook 08. All runs used the same T4 GPU type.

In [7]:
pd.DataFrame([
    {
        "setting": s,
        "input tokens": round(np.mean([a["input_tokens"] for a in answers[s].values()])),
        "answer tokens": round(np.mean([a["tokens"] for a in answers[s].values()])),
        "seconds": round(np.mean([a["seconds"] for a in answers[s].values()]), 1),
    }
    for s in SETTINGS
])

,setting,input tokens,answer tokens,seconds
0,gold_text,1261,493,38.1
1,gold_image,2728,403,33.4
2,retrieved_text,5204,573,45.6
3,retrieved_image,10754,415,47.2
4,closed_book,97,653,47.8


Images need about twice the input tokens of the same pages as text. Image answers are shorter, so the time per question is similar. 

## 7. Comparison with published results

Two published results are used as reference points. However, neither is measured under the same conditions, so they are not a direct ranking.

- *Islam et al. (2023)*, the FinanceBench paper, grade GPT-4-Turbo by hand on the 150 questions (their Table 2). Their oracle gives the model the evidence pages, or the full financial statements for metrics questions. Their shared vector store searches all 360 FinanceBench reports, while ours searches the 84 reports of the 150 questions. Closed book gives no documents. Their numbers include the 30 dev questions, and their prompt does not ask for step-by-step working.

- *Kobeissi and Langlais (2026)* report a numeric match on the 50 metrics questions with Qwen-2.5-7B (their Table 5), using the same 3% rule as our code labels. Our numbers below use that rule alone on the 40 metrics test questions, without the hand-checked labels.

In [8]:
# Islam et al. (2023), Table 2: GPT-4-Turbo, share of correct answers on the 150 questions.
financebench = {"oracle": 0.85, "shared vector store": 0.19, "closed book": 0.09}
matching = {"gold_text": "oracle", "gold_image": "oracle", "retrieved_text": "shared vector store", "retrieved_image": "shared vector store", "closed_book": "closed book"}

pd.DataFrame([
    {"setting": s, "Islam et al. setting": matching[s], "GPT-4-Turbo": financebench[matching[s]], "ours": round(correct[s].mean(), 3)}
    for s in SETTINGS
])

,setting,Islam et al. setting,GPT-4-Turbo,ours
0,gold_text,oracle,0.85,0.783
1,gold_image,oracle,0.85,0.800
2,retrieved_text,shared vector store,0.19,0.558
3,retrieved_image,shared vector store,0.19,0.567
4,closed_book,closed book,0.09,0.183


With the evidence pages, our reader model comes close to GPT-4-Turbo. With retrieved pages it is far higher, though their search covers 360 reports against our 84. In the closed book setting our reader also scores higher, mainly because it guesses where GPT-4-Turbo refused.

In [9]:
# Kobeissi and Langlais (2026), Table 5: numeric match on the 50 metrics questions, Qwen-2.5-7B.
published = {"oracle page": 0.34, "BGE-M3 + Multi-HyDE + reranker": 0.12, "learned page scorer (their method)": 0.30}

metrics = [fid for fid in test_ids if questions[fid]["question_type"] == "metrics-generated"]
ours = {s: round(np.mean([numeric_match(answers[s][fid]["final"] or "", questions[fid]["answer"]) for fid in metrics]), 3) for s in SETTINGS}

display(pd.Series(published, name="numeric match, Kobeissi and Langlais"))
pd.Series(ours, name=f"numeric match, ours ({len(metrics)} questions)")

oracle page                           0.34
BGE-M3 + Multi-HyDE + reranker        0.12
learned page scorer (their method)    0.30
Name: numeric match, Kobeissi and Langlais, dtype: float64

gold_text          0.875
gold_image         0.875
retrieved_text     0.650
retrieved_image    0.625
closed_book        0.025
Name: numeric match, ours (40 questions), dtype: float64

With the evidence pages, our numeric match on the metrics questions is far above their oracle page result. One difference is the prompt. Theirs asks for the answer directly, while ours asks for step-by-step working.

### Summary of Results

- Text and image give the same accuracy within noise, with the evidence pages and with retrieved pages (sections 1 and 2).
- Images cost about twice the input tokens without being more accurate (section 6).
- Search is the main source of errors. Retrieved pages lower accuracy reliably, and most misses happen when the evidence was not retrieved (sections 3 and 4).
- Without pages the model answers few questions correctly, so the other settings rely on the pages (section 1).
- With the evidence pages, the reader comes close to GPT-4-Turbo in the FinanceBench paper (section 7).